### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [24]:
# Librerías para manejo de datos
using CSV
using DataFrames
using Glob

# Librerías estadísticas y de procesamiento
using Statistics
using StatsBase
using Random

# Librerías para visualización
using PrettyTables

# Librerías para aprendizaje automático
using MLJ
using MLJModelInterface
using MultivariateStats
import MLJMultivariateStatsInterface
import MLJLIBSVMInterface
using NearestNeighborModels
using LIBSVM
using Flux
import MLJFlux

In [25]:
const MMI = MLJModelInterface
PCA = @load PCA pkg=MultivariateStats;
ICA = @load ICA pkg=MultivariateStats;
LDA = @load LDA pkg=MultivariateStats;
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels;
SVC = @load SVC pkg=LIBSVM;
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux;

import MLJMultivariateStatsInterface ✔
import MLJMultivariateStatsInterface ✔
import MLJMultivariateStatsInterface ✔
import NearestNeighborModels ✔
import MLJLIBSVMInterface

┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159
┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159


 ✔
import MLJFlux ✔


┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Pc\.julia\packages\MLJModels\BfLy4\src\loading.jl:159


# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

- ### Creamos un dataset único, usando los datos de las carpetas "Investigador A" e "Investigador B", cuyo contenido son ficheros .csv con datos. A su vez, mostramos información descriptiva sobre el dataset, como el número de variables,...

In [3]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


## 2. Análisis de valores ausentes

### Para analizar la calidad del dataset, se evaluó la presencia de valores ausentes tanto a nivel global como por variable. El porcentaje total de celdas con valores nulos en el dataset (`df_total`) es cercano al **1%** A nivel de columnas, **175 de las 563 variables** presentan al menos un valor faltante. Las variables con mayor proporción de valores ausentes alcanzan ligeramente más del **10%**.

In [4]:
function nulls_analysis(df::DataFrame; top=10)

    # Porcentaje total de nulos
    total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
    total_values  = nrow(df) * ncol(df)
    pct_total = (total_missing / total_values) * 100
    println("Porcentaje de valores ausentes en el dataset: $(pct_total) %")

    # Porcentaje de nulos por columna
    df_nulls = DataFrame(
        Variable = names(df),
        NullsPercentage = [mean(ismissing.(df[!, c])) * 100 for c in names(df)]
    )

    # Variables con algún valor nulo
    df_nulls_pos = filter(:NullsPercentage => p -> p > 0, df_nulls)
    println("Variables con algún valor ausente: ", nrow(df_nulls_pos))

    # Ordenar de mayor a menor
    sort!(df_nulls_pos, :NullsPercentage, rev=true)

    # Mostrar top variables con más nulos
    if nrow(df_nulls_pos) > 0
        n_show = min(top, nrow(df_nulls_pos))
        println("\nTop $n_show variables con valores ausentes:")
        pretty_table(first(df_nulls_pos, n_show))
    else
        println("Ninguna columna contiene valores ausentes.")
    end

    return df_nulls_pos, pct_total
end

nulls_analysis(df_total);

Porcentaje de valores ausentes en el dataset: 0.9984242033534787 %
Variables con algún valor ausente: 175

Top 10 variables con valores ausentes:
┌──────────────────────────┬─────────────────┐
│                 Variable │ NullsPercentage │
│                   String │         Float64 │
├──────────────────────────┼─────────────────┤
│       tBodyGyroMag-mad() │         10.0301 │
│       tBodyGyroMag-iqr() │         10.0301 │
│         fBodyAcc-mad()-Y │         10.0204 │
│    fBodyAccJerk-mean()-X │         10.0204 │
│  tBodyAccJerk-energy()-X │          10.001 │
│ tBodyAccJerk-entropy()-Y │          10.001 │
│        tBodyAccMag-max() │          10.001 │
│     tGravityAccMag-std() │          10.001 │
│ tGravityAccMag-entropy() │          10.001 │
│         fBodyAcc-std()-X │          10.001 │
└──────────────────────────┴─────────────────┘


## 3. Tratamiento y transformación de datos

### En esta sección preparamos el conjunto de datos para su uso en los algoritmos de clasificación. El objetivo es obtener un dataset completamente limpio, sin valores ausentes y con la variable objetivo correctamente codificada. Queremos mantener una versión sin modificar del dataset, por lo que creamos una copia independiente llamada **df_imputed**. Como los datos pertenecen a diferentes individuos, la imputación se realiza por sujeto.

In [5]:
df_imputed = deepcopy(df_total)

function impute_feature(df::DataFrame)
    individuals = groupby(df, :subject)

    for individual in individuals
        for col in names(individual)

            if col in (:subject, :Activity)
                continue
            end
        
            col_data = individual[!, col]

            if eltype(skipmissing(col_data)) <: Number #  Solo imputamos variables numéricas
                med = median(skipmissing(col_data))
                replace!(col_data, missing => med)
            end
        end
    end

    return df
end

impute_feature(df_imputed)
nulls_analysis(df_imputed); # Verificamos que ya no hay nulos

# La etiqueta debe ser categórica
df_imputed.Activity = categorical(df_imputed.Activity)

# Separamos features y target (características y etiqueta)
y = df_imputed.Activity
x = DataFrames.select(df_imputed, Not([:subject, :Activity]))

# Mostramos que estos tratamientos se aplican correctamente
println("\nTipo de la variable objetivo (y): $(eltype(df_imputed.Activity))")
println("Número de features (columnas en dataset de features): ", ncol(x))

Porcentaje de valores ausentes en el dataset: 0.0 %
Variables con algún valor ausente: 0
Ninguna columna contiene valores ausentes.

Tipo de la variable objetivo (y): CategoricalArrays.CategoricalValue{String31, UInt32}
Número de features (columnas en dataset de features): 561


## 4. Partición Holdout

### Para garantizar una evaluación estrictamente independiente, se realiza una partición hold-out basada en sujetos completos. El 10% de los individuos se reserva como test, y el 90% restante se usa para entrenamiento + validación. Esta división evita que datos del mismo sujeto aparezcan en distintos conjuntos, eliminando cualquier riesgo de data leakage.

In [6]:
subjects = unique(df_imputed.subject) # Sujetos únicos
Random.seed!(104)
shuffle!(subjects)
n_test = round(Int, length(subjects) * 0.10) # 10% de sujetos para test

test_subjects = subjects[1:n_test]
trainval_subjects = subjects[n_test+1:end]

df_trainval = filter(row -> row.subject in trainval_subjects, df_imputed); 
df_test = filter(row -> row.subject in test_subjects, df_imputed);

println("Sujetos en TEST (df_test): ", unique(df_test.subject))
println("Sujetos en TRAIN+VAL (df_trainval): ", unique(df_trainval.subject))

Sujetos en TEST (df_test): [25, 18, 22]
Sujetos en TRAIN+VAL (df_trainval): [1, 5, 7, 11, 3, 9, 23, 15, 17, 21, 13, 19, 27, 29, 2, 4, 6, 8, 10, 12, 14, 16, 20, 24, 26, 28, 30]


## 5. Cross-Validation individual wise

### Implementamos una validación cruzada por individuo (para evitar data leakage). Obtenemos la lista de los sujetos del dataset, los barajamos (para añadir aleatoriedad) y los asignamos a folds. La función devuelve un vector de longitud k, donde cada elemento es una tupla, formada cada una por 2 vectores, uno con los índices de entrenamiento y otro con los índices de validación.

In [7]:
function subject_folds(df::DataFrame; k::Int=5, seed::Int=104)
    subjects = unique(df.subject)
    Random.seed!(seed)
    shuffle!(subjects)

    # Asignamos un fold a cada sujeto, de forma más o menos equilibrada
    fold_id = Dict{eltype(subjects), Int}()
    for (i, s) in enumerate(subjects)
        fold_id[s] = 1 + (i - 1) % k
    end

    folds = Vector{Tuple{Vector{Int}, Vector{Int}}}(undef, k)

    for fold in 1:k # Construimos los folds
        train_idx = Int[]
        val_idx   = Int[]
        for (i, row) in enumerate(eachrow(df))
            if fold_id[row.subject] == fold
                push!(val_idx, i)
            else
                push!(train_idx, i)
            end
        end
        folds[fold] = (train_idx, val_idx)
    end

    return folds
end;

# 6. Normalización Min-Max

In [8]:
# Definición del modelo (wrapper)
struct MyMinMaxScaler <: MMI.Unsupervised
end

# Fase de entrenamiento (aprende min y max)
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)
    
    # Convertimos X a matriz
    Xmat = MMI.matrix(X)

    # Por columna: min y max
    mins = mapslices(minimum, Xmat; dims=1)[1, :]
    maxs = mapslices(maximum, Xmat; dims=1)[1, :]

    # Guardamos en cache
    cache = (mins = mins, maxs = maxs)
    report = nothing

    return cache, report
end

# Transformación: aplicar (X - min) / (max - min)
function MMI.transform(model::MyMinMaxScaler, cache, X)
    Xmat = MMI.matrix(X)
    mins = cache.mins
    maxs = cache.maxs

    # Evitar división por cero
    ranges = maxs .- mins
    ranges[ranges .== 0] .= 1  # columnas constantes

    Xscaled = (Xmat .- mins) ./ ranges

    return Xscaled
end

# Modelos básicos y selección de atributos (20%)

- # Helpers para los wrappers

### Definimos varias funciones auxiliares (lógica matemática,...) que nos ayudan a conseguir un código más limpio y reutilizable en los wrappers. Son las siguientes:
- ### '_y_to_int' Convierte una etiqueta categórica a un vector de enteros, necesario en cálculos de correlaciones, que requieren representación numérica.
- ### '_anova_f_score' Calcula el F-score para una feature (fórmula clásica de ANOVA). Un F-score alto indica que la feature separa bien las clases y, por tanto, es útil para clasificación.
- ### '_mutual_info' Calcula mutual information. Como la variable X es continua, se divide en intervalos (bins) uniformes. Se discretiza y se calcula el MI. Un valor alto de MI indica que la feature comparte mucha información con la variable objetivo.
- ### '_select_top_features' Selecciona las mejores características de un vector de scores, calculados por cualquiera de los filtrados.

In [9]:
# Pasar etiqueta categórica a enteros
function _y_to_int(y)
    unique_classes = levels(y) # Obtenemos una lista donde cada elemento es una de las etiquetas

    idx = Dict{eltype(unique_classes), Int}() # Diccionario clase -> entero
    for (i, class) in enumerate(unique_classes)
        idx[class] = i
    end
    
    classes = Int[] # Vector de enteros
    for v in y
        push!(classes, idx[v]) # Añadimos idx[v] a classes
    end

    return classes
end

# ANOVA F-score para una columna numérica y una etiqueta categórica
function _anova_f_score(x::AbstractVector{<:Real}, y)
    n = length(x) # Número total de observaciones
    classes = levels(y)
    C = length(classes) # Número de grupos

    if C < 2 # Si sólo hay 1, no tiene sentido el filtrado
        return 0.0
    end

    global_mean = mean(x) # Media de todas las observaciones
    SST = 0.0 # Variabilidad entre grupos
    SSE = 0.0 # Variabilidad dentro del grupo

    for class in classes
        mask = y .== class # Vector de booleanos para separar clases 
        xi = x[mask] # Observaciones de esa clase
        ni = length(xi)

        if ni == 0
            continue
        end

        m = mean(xi) # Media de la clase 
        SST += ni * (m - global_mean)^2 
        SSE += sum((xi .- m) .^ 2)
    end

    F = (SST / (C - 1)) / (SSE / (n - C))
    return F
end

# Mutual information (discretizando X en bins uniformes)
function _mutual_info(x::AbstractVector{<:Real}, y_int::Vector{Int}; nbins::Int=10)
    n = length(x) # Número total de observaciones
    if n == 0
        return 0.0
    end

    xmin, xmax = minimum(x), maximum(x) 
    if xmin == xmax # Si todos los valores de la feature son iguales, no aporta información
        return 0.0
    end

    # Dividimos el rango [xmin, xmax] en nbins intervalos iguales
    edges = range(xmin, xmax; length = nbins+1)

    xbin = Vector{Int}(undef, n) # En este vector asignaremos cada valor de X a un bin
    for i in 1:n
        v = x[i]
        if v == xmax # Si el valor actual es el máximo, lo forzamos al último bin.
            xbin[i] = nbins
        else
            frac = (v - xmin) / (xmax - xmin)
            b = floor(Int, frac * nbins) + 1 # Aquí convertimos el valor normalizado anteriormente en un índice de bin. floor redondea al entero más bajo.

            if b < 1
                b = 1
            end

            if b > nbins
                b = nbins
            end

            xbin[i] = b
        end
    end

    num_classes = maximum(y_int)
    counts = zeros(Float64, nbins, num_classes) # Matriz de 0, con nbins filas y num_classes columnas

    for i in 1:n
        counts[xbin[i], y_int[i]] += 1
    end

    pxy = counts ./ n # Probabilidad conjunta
    px = vec(sum(pxy, dims=2)) # Probabilidad marginal de las filas
    py = vec(sum(pxy, dims=1)) # Probabilidad de las columnas

    mi = 0.0
    for i in 1:nbins
        for j in 1:k
            pij = pxy[i, j]
            if pij > 0.0
                mi += pij * log(pij / (px[i] * py[j]))
            end
        end
    end

    return mi
end

# Selecciona top-n según scores
function _select_top_features(scores::AbstractVector{<:Real}, n_features::Int)
    p = length(scores)
    n_sel = min(n_features, p)
    idx_sorted = sortperm(scores, rev=true)
    return idx_sorted[1:n_sel]
end;

- # Filtrado ANOVA

- ### El filtrado ANOVA evalúa cada feature midiendo cuánto separa las clases mediante el F-score, que compara la variabilidad entre grupos y dentro de los grupos. un valor alto indica que la feature discrimina bien entre clases.

In [10]:
# Wrapper de ANOVA
struct MyANOVAFilter <: MMI.Supervised # Definimos modelo MLJ que hereda de MMI.Supervised, con un solo parámetro (n_features)
    n_features::Int
end

MyANOVAFilter(; n_features::Int=100) = MyANOVAFilter(n_features) # Constructor del filtro

function MMI.fit(model::MyANOVAFilter, verbosity::Int, X, y) # Función fit. Recibe instancia de MyANOVAFilter
    Xmat = MMI.matrix(X)
    y_cat = coerce(y, Multiclass)  # Aseguramos que y es un CategoricalVector
    n, p = size(Xmat)

    scores = zeros(Float64, p) # un hueco por feature
    for j in 1:p
        scores[j] = _anova_f_score(view(Xmat, :, j), y_cat) # para cada columna j, calcula el F-score con respecto a y_cat
    end

    selected = _select_top_features(scores, model.n_features)
    cache = (selected = selected, scores = scores) # MLJ guarda esto tras el fit
    report = nothing
    return cache, report
end

function MMI.transform(model::MyANOVAFilter, cache, X) # Transformación. Devuelve el dataset reducido con los scores selected
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # Filtrado Pearson

-  ### Calcula la correlación lineal entre cada feature y la variable objetivo, mediante el coeficiente de Pearson. Se usa el valor absoluto de la correlación porque las relaciones fuertes pueden ser tanto positivas como negativas.

In [11]:
# Wrapper de Pearson
struct MyPearsonFilter <: MMI.Supervised
    n_features::Int
end

MyPearsonFilter(; n_features::Int=100) = MyPearsonFilter(n_features) # Constructor

function MMI.fit(model::MyPearsonFilter, verbosity::Int, X, y) # Función fit
    Xmat = MMI.matrix(X)
    y_int = _y_to_int(coerce(y, Multiclass)) # Convertimos la variable objetivo a numérica
    y_float = Float64.(y_int)

    n, p = size(Xmat)
    scores = zeros(Float64, p)

    for j in 1:p
        xj = view(Xmat, :, j)
        scores[j] = abs(cor(xj, y_float)) # El valor absoluto nos ayuda a tener en cuenta las correlaciones negativas altas.
    end

    selected = _select_top_features(scores, model.n_features)
    cache = (selected = selected, scores = scores)
    report = nothing
    return cache, report
end

function MMI.transform(model::MyPearsonFilter, cache, X) # Transform
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # Filtrado Spearman

- ### Mide la correlación monotónica entre cada feature y la variable objetivo, comparando sus rangos en lugar de los valores originales. Es útil cuando la relación no es lineal, pero sí creciente o decreciente.

In [12]:
# Wrapper de Spearman
struct MySpearmanFilter <: MMI.Supervised
    n_features::Int
end

MySpearmanFilter(; n_features::Int=100) = MySpearmanFilter(n_features) # Constructor

function MMI.fit(model::MySpearmanFilter, verbosity::Int, X, y) #  Función fit
    Xmat = MMI.matrix(X)
    y_int = _y_to_int(coerce(y, Multiclass))
    y_float = Float64.(y_int)

    n, p = size(Xmat)
    scores = zeros(Float64, p)

    for j in 1:p
        xj = view(Xmat, :, j)
        scores[j] = abs(corspearman(xj, y_float)) # Correlación de Spearman
    end

    selected = _select_top_features(scores, model.n_features)
    cache = (selected = selected, scores = scores)
    report = nothing
    return cache, report
end

function MMI.transform(model::MySpearmanFilter, cache, X) # Transform
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # Filtrado Kendall Tau

- ### Evalúa la relación entre feature y etiqueta contando pares concordantes y discordantes, calculando el coeficiente τ de Kendall. τ alto implica una relación monotónica fuerte (más concordancias que discordancias). Tomamos |τ| como score de importancia.

In [13]:
# Wrapper Kendall Tau
struct MyKendallFilter <: MMI.Supervised
    n_features::Int
end

MyKendallFilter(; n_features::Int=100) = MyKendallFilter(n_features) # Constructor

function MMI.fit(model::MyKendallFilter, verbosity::Int, X, y)
    Xmat = MMI.matrix(X)
    y_int = _y_to_int(coerce(y, Multiclass)) # Variable objetivo a numérica
    y_float = Float64.(y_int)

    n, p = size(Xmat)
    scores = zeros(Float64, p)

    for j in 1:p
        xj = view(Xmat, :, j)
        scores[j] = abs(corkendall(xj, y_float))
    end

    selected = _select_top_features(scores, model.n_features)
    cache = (selected = selected, scores = scores)
    report = nothing
    return cache, report
end

function MMI.transform(model::MyKendallFilter, cache, X) # Transform
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # Mutual Information

- ### Mide cuánta información comparte cada feature con la etiqueta. Un valor de MI alto significa que conocer esa feature reduce la incertidumbre sobre la clase, por lo que es muy informativa.

In [14]:
# Wrapper Mutual Information
struct MyMIFilter <: MMI.Supervised
    n_features::Int
end

MyMIFilter(; n_features::Int=100) = MyMIFilter(n_features) # Constructor

function MMI.fit(model::MyMIFilter, verbosity::Int, X, y) # Función fit
    Xmat = MMI.matrix(X)
    y_int = _y_to_int(coerce(y, Multiclass))

    n, p = size(Xmat)
    scores = zeros(Float64, p)

    for j in 1:p
        xj = view(Xmat, :, j)
        scores[j] = _mutual_info(xj, y_int)
    end

    selected = _select_top_features(scores, model.n_features)
    cache = (selected = selected, scores = scores)
    report = nothing
    return cache, report
end

function MMI.transform(model::MyMIFilter, cache, X) # Transform
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # RFE

- ### RFE elimina recursivamente las características menos relevantes según un modelo entrenado (normalmente lineal). En cada iteración se entrena un modelo sobre las features actuales y se mide su importancia mediante los pesos |w|: un peso pequeño indica baja contribución al modelo. Se elimina la feature con menor importancia y se repite el proceso hasta que nos quedamos con las n_features deseadas.

In [15]:
# Wrapper para RFE
struct MyRFEFilter <: MMI.Supervised
    n_features::Int
end

MyRFEFilter(; n_features::Int=100) = MyRFEFilter(n_features) # Constructor

function MMI.fit(model::MyRFEFilter, verbosity::Int, X, y) # Función fit
    Xmat = MMI.matrix(X)
    y_int = _y_to_int(coerce(y, Multiclass))
    y_float = Float64.(y_int)
    
    n, p = size(Xmat)
    scores = zeros(Float64, p)

    chosen = collect(1:p)

    while length(remaining) > model.n_features
        Xi = Xmat[:, remaining] # Columna actual
        w  = Xi \ y_float # Pesos por mínimos cuadrados
        value = abs.(w)

        j_min = argmin(value) # Posición del peso más pequeño de chosen
        deleteat!(remaining, j_min) # Eliminamos esa feature
    end

    selected = remaining
    cache = (selected = selected, scores = scores)
    report = nothing
    return cache, report
end

function MMI.transform(model::MyRFEFilter, cache, X) # Transform
    Xmat = MMI.matrix(X)
    return Xmat[:, cache.selected]
end

- # Reducción de dimensionalidad PCA

In [26]:
# Carga de datos y partición
Xtrainval = deepcopy(df_trainval[:, Not(:subject, :Activity)])
ytrainval = df_trainval.Activity
Xtest = deepcopy(df_test[:, Not(:subject, :Activity)])
ytest = df_test.Activity

# Creación y ajuste del modelo PCA
pca_model = PCA(maxoutdim=20)
pca_mach = fit(pca_model, Xtrainval)

# Transformación de los datos
Xtrain_pca = transform(pca_mach, Xtrainval)
Xtest_pca = transform(pca_mach, Xtest)

UndefVarError: UndefVarError: `fit` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name may be made accessible by importing StatsAPI in the current active module Main
Hint: a global variable of this name also exists in StatsBase.
Hint: a global variable of this name may be made accessible by importing Distributions in the current active module Main
Hint: a global variable of this name may be made accessible by importing LearnAPI in the current active module Main
Hint: a global variable of this name also exists in MLJModelInterface.
Hint: a global variable of this name may be made accessible by importing MLJBase in the current active module Main
Hint: a global variable of this name also exists in MultivariateStats.

- # Reducción de dimensionalidad ICA


In [ ]:
ica_model = ICA(maxoutdim=20)

ica_mach = machine(ica_model, Xtrain)
fit!(ica_mach)

Xtrain_ica = transform(ica_mach, Xtrain)
Xtest_ica  = transform(ica_mach, Xtest)

- # Reducción de dimensionalidad LDA


In [ ]:
lda_model = LDA()

lda_mach = machine(lda_model, Xtrain, ytrain)
fit!(lda_mach)

Xtrain_lda = transform(lda_mach, Xtrain)
Xtest_lda  = transform(lda_mach, Xtest)

- # MLP

- # SVM

- # KNN